# 00 - Checagem de ambiente

Confirma, em um único teste: Java + PySpark sobem, os jars do Delta Lake resolvem, a conexão S3A com o MinIO funciona, e dá para escrever/ler uma tabela Delta no bucket `cairn-bolt-lakehouse`.

Rode esta célula uma vez. Na primeira execução o Spark baixa os jars do Delta/Hadoop-AWS via Maven, pode levar 1-2 minutos.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("cairn-bolt-smoke-test")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4", "com.amazonaws:aws-java-sdk-bundle:1.12.262"]
).getOrCreate()

print("Spark version:", spark.version)

df = spark.createDataFrame([(1, "ok"), (2, "delta+minio funcionando")], ["id", "mensagem"])
df.write.format("delta").mode("overwrite").save("s3a://cairn-bolt-lakehouse/bronze/_smoke_test")

check = spark.read.format("delta").load("s3a://cairn-bolt-lakehouse/bronze/_smoke_test")
check.show()